# Classificação Hiperespectral com MLOps

Um único notebook, autocontido, para otimizar e avaliar uma CNN 1D em imagens hiperespectrais. O roteiro segue o fluxo real de trabalho: configurar, preparar os dados, treinar, rastrear e observar os resultados.

## Como usar este notebook

Para executar um experimento, rode as seções **1 a 7** em ordem e, depois, habilite `RUN_EXPERIMENT` na seção 8. O código está no pacote `hsi/` (arquivos `.py`, um por responsabilidade); as seções 1 a 7 só importam dele, e cada célula de import diz em que arquivo está o código daquela seção. Para entender a organização, comece por `hsi/__init__.py`, que tem o mapa do pacote.

- Dados esperados: arquivos `.mat` em `data/` (por padrão, Indian Pines).
- Dependências: `requirements.txt` (versões pinadas e conferidas por `check_versions()` na seção 2). Instale com `uv pip install -r requirements.txt` e depois `uv pip install -e . --no-deps` (o pacote `hsi`).
- Saídas: `outputs/runs/` contém métricas, configuração, figuras, checkpoints e manifestos; `outputs/registry/` contém os modelos promovidos.

### Roteiro de navegação

1. **Base e configuração** — seções 1 e 2: caminhos, ambiente, GPU, determinismo e meta de treinamento.
2. **Dados e segmentação** — seção 3: PCA, segmentadores, diagnóstico visual e glossário.
3. **Modelos e treino** — seções 4 e 5: CNN 1D, early stopping, interpretabilidade e HPO.
4. **Rastreabilidade** — seções 6 e 7: artefatos, campanhas e variantes comparáveis.
5. **Execução** — seção 8: habilite somente o painel/campanha desejado.
6. **Resultados** — seção 9: histórico auditável e comparação entre execuções.


## 1. Base do experimento

### 1.1 Caminhos, contrato de configuração e ambiente

Caminhos e configuração tipada em `hsi/config.py`; versões, GPU e determinismo em `hsi/ambiente.py`. Em geral não precisam ser alterados: a configuração prática está na próxima seção.

In [ ]:
# Edições nos arquivos de `hsi/` passam a valer sem reiniciar o kernel.
%load_ext autoreload
%autoreload 2

# O código desta seção está em:
#     hsi/ambiente.py
#     hsi/config.py

import matplotlib.pyplot as plt

from hsi.config import ExperimentConfig, MODEL_TYPES
from hsi.ambiente import (
    check_versions, configure_determinism, seed_everything, select_device,
)

## 2. Escolher ambiente e configuração

### 2.1 Ambiente, GPU e reprodutibilidade

A configuração abaixo é o principal ponto de ajuste. Comece com poucas épocas para validar o fluxo; depois aumente `epochs` e ajuste `batch_size` conforme a memória da GPU.

### 2.2 Meta de treinamento e early stopping

**Meta atual:** atingir `val_accuracy ≥ 0,95`. Caso a meta não seja alcançada, o treino é encerrado depois de **10 épocas** sem melhora de pelo menos **0,001** (0,1 ponto percentual) em `val_accuracy`. A validação governa tanto a meta quanto a parada; o teste permanece reservado para o relato final. Ajuste `early_stopping_target`, `patience` e `early_stopping_min_delta` juntos.


In [ ]:
check_versions()
device = select_device(verbose=True)
configure_determinism(deterministic=True)
seed_everything(seed=42)

config = ExperimentConfig(
    dataset='IP',
    segmenter='seeds',          # opções: seeds, slic, lsc
    two_input=True,             # concatena espectro e segmento no CNN1D
    pca_components=30,
    batch_size=128,
    epochs=60,
    # Meta: 95% na validação OU 10 épocas sem ganho de 0,1 p.p.
    early_stopping_monitor='val_accuracy',
    early_stopping_target=0.95,
    patience=10,
    early_stopping_min_delta=0.001,
    restore_best_weights=True,
    num_workers=0,              # aumente apenas após validar o SO/ambiente
)
print(f'Dispositivo: {device} | arquitetura: {config.model_type} | segmentador: {config.segmenter}')


## 3. Preparar os dados e o contexto espacial

### 3.1 Carregamento, normalização e divisões

A preparação transforma a cena em amostras rotuladas, reduz as bandas com PCA e cria superpixels. A função `prepare_data_parallel` permite executar PCA e segmentação em paralelo por threads ou processos antes da fusão.

In [ ]:
# O código desta seção está em:
#     hsi/dados/cenas.py
#     hsi/dados/particoes.py
#     hsi/dados/preprocessamento.py
#     hsi/dados/segmentacao.py
#     hsi/paralelo/ramos.py

from hsi.dados.cenas import load_scene

### 3.2 Segmentadores disponíveis

Quatro algoritmos: **SEEDS**, **SLIC**, **LSC** (superpixels do `cv.ximgproc`) e **Watershed** (bacias de gradiente do OpenCV base). Todos recebem a mesma composição PCA/RGB e devolvem `SegmentationResult` com labels e contornos no mesmo formato, então trocar de segmentador não altera DataLoader, CNN nem métricas.

Os parâmetros por cena ficam em `SEEDS_PARAMS`, `SLIC_PARAMS`, `LSC_PARAMS` e `WATERSHED_PARAMS`; `config.segmenter_overrides` sobrescreve campos individuais e é o que o HPO varia.

### 3.3 Explorar a cena e comparar segmentadores

Antes de treinar, confirme visualmente a distribuição de classes, assinaturas espectrais, variância do PCA e regiões produzidas por cada superpixel. A pureza aparece em um gráfico de homogeneidade espacial — não usa nem substitui as métricas do classificador.

In [ ]:
# O código desta seção está em:
#     hsi/visualizacao/diagnosticos.py

from hsi.visualizacao.diagnosticos import (
    plot_evaluation_dashboard, plot_inference_profile_chart, plot_parallel_profile,
    plot_pca_diagnostics, plot_per_class_metrics, plot_runs_dashboard,
    plot_scene_overview, plot_segment_metrics_visual, plot_segmentation_comparison,
    plot_segmentation_overlay_suite, plot_segmentation_quality_dashboard,
    profile_inference,
)

### 3.4 Glossário de leitura dos gráficos

Os painéis apresentam métricas de qualidade, incerteza, explicabilidade e desempenho. Para evitar que um gráfico seja interpretado apenas pela cor ou pelo título, `hsi/visualizacao/glossario.py` cria uma legenda contextual: cada resultado exibido nas seções 3 e 8 mostra os termos pertinentes e como interpretá-los. **Confiança alta não garante acerto**; ela é a probabilidade máxima atribuída pelo modelo. **Entropia alta** indica distribuição de probabilidades mais dividida entre as classes e, portanto, maior incerteza.


In [ ]:
# O código desta seção está em:
#     hsi/visualizacao/glossario.py

from hsi.visualizacao.glossario import show_glossary

In [ ]:
RUN_DATA_DIAGNOSTICS = True

if RUN_DATA_DIAGNOSTICS:
    scene_for_visuals = load_scene(config.dataset, download=False)
    display(plot_scene_overview(scene_for_visuals))
    pca_figure, _, _ = plot_pca_diagnostics(scene_for_visuals)
    display(pca_figure)
    segmentation_figure, _ = plot_segmentation_comparison(scene_for_visuals)
    display(segmentation_figure)
    display(plot_segmentation_overlay_suite(scene_for_visuals))
    display(plot_segmentation_quality_dashboard(scene_for_visuals))
    show_glossary('dados')
else:
    print('Diagnósticos de dados desativados.')

# As figuras já foram exibidas com display(); fechá-las impede que o Jupyter
# as mostre de novo ao fim da célula.
plt.close('all')


### 3.5 Métricas usadas e justificativa

O seu TCC define como núcleo **OA**, **AA**, **Kappa**, tempos, memória, parâmetros e speedup. OA mede o acerto total; AA é a média das revocações por classe e protege as classes raras; Kappa corrige a concordância esperada ao acaso. Este notebook acrescenta precisão, F1 macro e IoU macro por classe como diagnóstico de erros assimétricos. A matriz de confusão e o mapa de erros tornam esses números localizáveis na cena.

Referências: [seção de métricas do TCC](tcc/TCC%20III%20-%20EDUARDO%20SAVIAN/src/2_fundamentacao.tex), [planejamento experimental do TCC](tcc/TCC%20III%20-%20EDUARDO%20SAVIAN/src/3_projeto.tex), [revisão de CNNs em HSI](https://www.sciencedirect.com/org/science/article/pii/S1526149222001825) e [documentação das métricas do scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html).

## 4. Modelos CNN

### 4.1 Entradas, convoluções e fusão

- **CNN1D:** explora o espectro de cada pixel; opcionalmente recebe o ID do superpixel.

O factory `build_model` seleciona o modelo a partir de `config.architecture`.

In [ ]:
# O código desta seção está em:
#     hsi/modelo.py

# (nenhuma função desta seção é chamada diretamente pelas células abaixo)

## 5. Treinar, avaliar e salvar

### 5.1 Laço de treino, validação e checkpoint

O treino usa batching na GPU, acumulação de gradientes quando necessário, parada antecipada e melhor checkpoint de validação. A avaliação registra OA, AA, kappa, métricas por classe e matriz de confusão.

In [ ]:
# O código desta seção está em:
#     hsi/avaliacao.py
#     hsi/checkpoint.py
#     hsi/treino.py
#     hsi/visualizacao/originais.py

# (nenhuma função desta seção é chamada diretamente pelas células abaixo)

### 5.2 Interpretabilidade visual das CNNs

Esta seção responde **o que a rede ativou** e **onde/por qual dimensão a decisão foi mais sensível**. Ela separa três ideias: ativações por filtro, redução feita por MaxPooling e atribuição à classe prevista via Grad-CAM/saliência. Nenhuma delas substitui OA, AA, Kappa ou validação experimental.

In [ ]:
# O código desta seção está em:
#     hsi/visualizacao/explicabilidade.py

from hsi.visualizacao.explicabilidade import (
    capture_activations, explanation_sample, grad_cam, plot_activation_atlases,
    plot_grad_cam, plot_input_saliency, plot_maxpool_effect,
    plot_prediction_uncertainty,
)

In [ ]:
# O código desta seção está em:
#     hsi/visualizacao/dashboards.py

from hsi.visualizacao.dashboards import plot_final_cnn_compilation

### 5.4 Otimização automática de hiperparâmetros

Três estratégias percorrem **o mesmo espaço**, com **o mesmo orçamento de trials** e **a mesma cobertura inicial** dos quatro segmentadores. A única diferença é como cada candidato é escolhido — é o que torna a comparação entre elas honesta.

| Estratégia | Como escolhe o próximo candidato |
| --- | --- |
| `random` | Sorteio uniforme, sem memória. É a linha de base. |
| `tpe` | Hipertunador bayesiano (Optuna). Modela a densidade dos trials bons e a dos ruins e propõe pontos que maximizam a razão entre elas. |
| `genetic` | Algoritmo genético elitista: seleção por torneio, crossover uniforme, mutação por gene e elite que passa sem retreinar. |

O espaço é totalmente discreto: `HPO_SPACE` define learning rate, dropout, weight decay, batch e componentes de PCA, e mais dois genes cobrem o segmentador e qual das três receitas internas dele usar. São 2.916 combinações para a CNN1D e 8.748 para as demais — grande demais para grade exaustiva, que é justamente o motivo de existirem estratégias de busca.

A representação comum é uma **tupla de índices**, um por gene (`hpo_genes` / `hpo_decode`). É ela que permite ao algoritmo genético fazer crossover sem codificação binária e ao TPE tratar o segmentador como categórico e os demais eixos como ordinais.

A seleção usa exclusivamente a validação; métricas de teste são relato, nunca critério. `compare_hpo_strategies` roda as três e `plot_hpo_strategies` mostra a curva de convergência — compare pelo **trial em que cada uma atinge um patamar**, não só pelo valor final: chegar ao mesmo lugar com metade dos treinos é o ganho real de um hipertunador.

In [ ]:
# O código desta seção está em:
#     hsi/hpo/busca.py
#     hsi/hpo/espaco.py
#     hsi/hpo/estrategias.py
#     hsi/hpo/graficos.py

from hsi.hpo.busca import automated_hpo, compare_hpo_strategies, hpo_strategy_summary
from hsi.hpo.graficos import (
    plot_hpo_by_cnn, plot_hpo_compilation, plot_hpo_space, plot_hpo_strategies,
)

### 5.5 Campanha MLOps auditável

O HPO exploratório sugere candidatos; esta campanha os repete com várias seeds e registra cada tentativa. O ranking usa validação média e IC95%; OA/AA/Kappa de teste permanecem no relatório para evitar a escolha indireta pelo teste.

In [ ]:
# O código desta seção está em:
#     hsi/mlops/campanha.py

from hsi.mlops.campanha import plot_campaign_analysis

In [ ]:
# O código desta seção está em:
#     hsi/mlops/relatorios.py

from hsi.mlops.relatorios import generate_standalone_pdf_report

In [ ]:
# O código desta seção está em:
#     hsi/mlops/paineis.py

from hsi.mlops.paineis import (
    all_run_figures, generate_all_runs_pdf, load_all_audited_runs, show_all_run_note,
)

In [ ]:
# O código desta seção está em:
#     hsi/mlops/paineis.py

# (nenhuma função desta seção é chamada diretamente pelas células abaixo)

### 5.6 Explicação inversa: da decisão para a imagem

Esta camada reconstrói previsões, confiança, entropia, acertos, erros e F1 no local onde ocorreram. Em seguida agrega essas evidências por superpixel e escolhe automaticamente detecções representativas para Grad-CAM.

In [ ]:
# O código desta seção está em:
#     hsi/visualizacao/explicacao_inversa.py

from hsi.visualizacao.explicacao_inversa import (
    display_cnn_dashboard, render_cnn_dashboard, render_spatial_explanation,
)

In [ ]:
# O código desta seção está em:
#     hsi/hpo/selecao.py

from hsi.hpo.selecao import best_hpo_results_by_cnn, run_best_hpo_campaign

In [ ]:
# O código desta seção está em:
#     hsi/hpo/varredura_batch.py

from hsi.hpo.varredura_batch import plot_batch_sweep, run_batch_sweeps_for_hpo_winners

## 6. MLOps: rastreabilidade, registro e observabilidade

### 6.1 Identidade da execução, artefatos e métricas

Cada execução recebe um identificador, salva configuração, versões, fingerprints dos dados, métricas, artefatos e manifesto. O registry promove somente runs concluídos; as funções de observabilidade consolidam comparações e alertas locais.

In [ ]:
# O código desta seção está em:
#     hsi/mlops/observabilidade.py
#     hsi/mlops/rastreio.py
#     hsi/mlops/registro.py

from hsi.mlops.observabilidade import discover_runs, filter_runs, runs_summary

## 7. Orquestrar as variantes

### 7.1 Receitas comparáveis e paralelismo

`run_experiment` liga todas as etapas: dispositivo, dados, loaders, modelo, treino e avaliação.

In [ ]:
# O código desta seção está em:
#     hsi/paralelo/particao.py
#     hsi/pipeline.py
#     hsi/variantes.py

from hsi.pipeline import run_experiment

## 8. Executar o experimento

### 8.0 Controle de execução

A execução pesada é intencionalmente protegida. Mude `RUN_EXPERIMENT` para `True` somente quando quiser treinar; assim, `Run All` pode ser usado para revisar e validar o notebook sem iniciar um treino de 60 épocas.

In [ ]:
RUN_EXPERIMENT = True

if RUN_EXPERIMENT:
    result = run_experiment(config, backend='thread', verbose=True)
    pdf_report = generate_standalone_pdf_report(result)
    print(f"\nRelatório PDF: {pdf_report}")
    print("Experimento concluído com sucesso.")
    print(result.metrics.summary())
else:
    print('Treino não iniciado. Defina RUN_EXPERIMENT = True para executar.')


### 8.1 Painel visual do resultado

Quando o treino estiver habilitado, este painel mostra convergência, métricas globais e por classe, matriz de confusão, mapa de erros, custo de inferência/memória, perfil dos ramos paralelos e a melhor época/motivo da parada antecipada.

In [ ]:
if RUN_EXPERIMENT:
    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    evaluation_figure, per_class_metrics, extra_metrics = plot_evaluation_dashboard(result, scene_for_visuals)
    display(evaluation_figure)
    display(plot_per_class_metrics(per_class_metrics))
    display(plot_inference_profile_chart(profile_inference(result)))
    display(plot_parallel_profile(result.prep_timings))
    show_glossary('avaliacao')
else:
    print('Execute o treino para gerar o painel de avaliação.')

# As figuras já foram exibidas com display(); fechá-las impede que o Jupyter
# as mostre de novo ao fim da célula.
plt.close('all')


### 8.2 O que a CNN está usando para decidir

O painel usa um pixel do conjunto de teste: os mapas são filtros × posições espectrais ao longo das bandas.

In [ ]:
if RUN_EXPERIMENT:
    x_explain, seg_explain, true_class = explanation_sample(result, split='test', batch_index=0)
    activations, predicted_class = capture_activations(result.model, x_explain, seg_explain)
    print(f'Classe real: {true_class + 1} | classe prevista: {predicted_class + 1}')

    for figure in plot_activation_atlases(activations).values():
        display(figure)
    for figure in plot_maxpool_effect(activations).values():
        display(figure)

    cam, cam_class, cam_layer = grad_cam(result.model, x_explain, seg_explain)
    display(plot_grad_cam(cam, result.config.model_type, cam_class, cam_layer))
    display(plot_input_saliency(result.model, x_explain, seg_explain, result.config.model_type))
    display(plot_prediction_uncertainty(result, scene_for_visuals))
    show_glossary('explicabilidade')
else:
    print('Execute o treino para gerar ativações, Grad-CAM e saliência.')

# As figuras já foram exibidas com display(); fechá-las impede que o Jupyter
# as mostre de novo ao fim da célula.
plt.close('all')


### 8.4 HPO por CNN e segmentador

Habilite esta campanha para buscar automaticamente a melhor combinação. Ela sempre cobre SEEDS, SLIC, LSC e Watershed; em seguida renderiza os outputs e o dashboard do vencedor.

In [ ]:
RUN_HPO = True
HPO_TRIALS_PER_CNN = 8      # mínimo 4: um candidato por segmentador
HPO_SEARCH_EPOCHS = 10      # orçamento reduzido para a busca; retreine vencedor depois
HPO_STRATEGY = 'tpe'        # 'random' | 'tpe' (bayesiano) | 'genetic'

# Compara as três estratégias com orçamento idêntico. Custa 3x o HPO normal,
# então é opt-in separado.
RUN_STRATEGY_COMPARISON = False
STRATEGY_TRIALS = 20

if RUN_HPO:
    hpo_table, hpo_winners = automated_hpo(
        config,
        trials_per_cnn=HPO_TRIALS_PER_CNN,
        search_epochs=HPO_SEARCH_EPOCHS,
        backend='thread',
        strategy=HPO_STRATEGY,
    )
    for cnn in MODEL_TYPES:
        hpo_figure, hpo_output = plot_hpo_by_cnn(hpo_table, cnn)
        display(hpo_figure)

    hpo_compilation, hpo_winner_table = plot_hpo_compilation(hpo_table)
    display(hpo_compilation)

    # O que a busca de fato variou, eixo a eixo.
    hpo_space_figure = plot_hpo_space(hpo_table)
    display(hpo_space_figure)
    plt.close(hpo_space_figure)

    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    hpo_dashboards = {name: render_cnn_dashboard(result, scene_for_visuals) for name, result in hpo_winners.items()}
    for name, dashboard in hpo_dashboards.items():
        display_cnn_dashboard(f'{name} — vencedor HPO', dashboard)
    hpo_final_figure, hpo_final_summary, _ = plot_final_cnn_compilation(hpo_dashboards)
    display(hpo_final_figure)
    plt.close(hpo_final_figure)
    show_glossary('hpo', 'comparacao')
else:
    print('HPO desativado. Defina RUN_HPO = True para otimizar cada CNN e mostrar seus outputs.')

if RUN_STRATEGY_COMPARISON:
    strategy_comparison = compare_hpo_strategies(
        config,
        trials_per_cnn=STRATEGY_TRIALS,
        search_epochs=HPO_SEARCH_EPOCHS,
        backend='thread',
        model_types=('cnn1d',),
    )
    display(hpo_strategy_summary(strategy_comparison))
    strategy_figure = plot_hpo_strategies(strategy_comparison)
    display(strategy_figure)
    plt.close(strategy_figure)

# As figuras já foram exibidas com display(); fechá-las impede que o Jupyter
# as mostre de novo ao fim da célula.
plt.close('all')


### 8.5 Campanha auditável com múltiplas execuções

Esta é a execução MLOps para resultados comparáveis: quatro receitas por CNN, quatro segmentadores e três seeds por padrão. Todos os JSONs, checkpoints, fingerprints e relatórios ficam em `outputs/campaigns/`, com links para os runs individuais em `outputs/runs/`.

In [ ]:
RUN_AUDITED_CAMPAIGN = True
# O orçamento de screening é o mesmo definido na seção 8.4: mudar lá muda aqui,
# em vez de duas seções discordarem silenciosamente sobre o tamanho da busca.
FINAL_HPO_SEEDS = (11, 29, 42)
FINAL_HPO_EPOCHS = config.epochs  # orçamento completo, só para vencedores HPO

if RUN_AUDITED_CAMPAIGN:
    # 1) Screening: cobre todos os segmentadores e escolhe por validação.
    hpo_table, _ = automated_hpo(
        config,
        trials_per_cnn=HPO_TRIALS_PER_CNN,
        search_epochs=HPO_SEARCH_EPOCHS,
        backend='thread',
    )
    # 2) Final: repete exclusivamente o melhor HPO de cada CNN, com várias seeds.
    campaign = run_best_hpo_campaign(
        hpo_table,
        final_epochs=FINAL_HPO_EPOCHS,
        seeds=FINAL_HPO_SEEDS,
        backend='thread',
    )
    print(f"Campanha dos melhores HPO: {campaign['directory']}")
    campaign_figure = plot_campaign_analysis(campaign['records'], campaign['analysis'])
    if campaign_figure is not None:
        display(campaign_figure)
        plt.close(campaign_figure)
    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    best_hpo_results = best_hpo_results_by_cnn(campaign)
    best_hpo_dashboards = {name: render_cnn_dashboard(result, scene_for_visuals) for name, result in best_hpo_results.items()}
    for name, dashboard in best_hpo_dashboards.items():
        display_cnn_dashboard(f'{name} — melhor HPO', dashboard)
    best_hpo_figure, best_hpo_summary, _ = plot_final_cnn_compilation(best_hpo_dashboards)
    display(best_hpo_figure)
    plt.close(best_hpo_figure)
    show_glossary('campanha', 'comparacao')
else:
    print('Campanha desativada. Quando habilitada, ela usa sempre o melhor HPO de cada CNN.')

# As figuras já foram exibidas com display(); fechá-las impede que o Jupyter
# as mostre de novo ao fim da célula.
plt.close('all')


### 8.6 Explicar o resultado diretamente na imagem

Use esta seção após treinar uma CNN individual. Ela remonta tudo no espaço da cena: segmentos, classes, métricas, erros, incerteza e exemplos Grad-CAM escolhidos automaticamente.

In [ ]:
if RUN_EXPERIMENT:
    if 'scene_for_visuals' not in globals():
        scene_for_visuals = load_scene(config.dataset, download=False)
    spatial_explanation = render_spatial_explanation(result, scene_for_visuals)
    for title, figure in spatial_explanation['figures'].items():
        print(title)
        display(figure)
        plt.close(figure)
    display(plot_segment_metrics_visual(spatial_explanation['segmentos']))
    show_glossary('espacial', 'explicabilidade')
else:
    print('Execute uma CNN na seção 8 antes de reconstruir suas decisões na imagem.')

# As figuras já foram exibidas com display(); fechá-las impede que o Jupyter
# as mostre de novo ao fim da célula.
plt.close('all')


### 8.7 HPO de segmentação + batches de 50 em 50

A busca escolhe primeiro o melhor HPO por CNN, incluindo os parâmetros do segmentador. Depois percorre uma **escada aritmética fixa**: 50, 100, 150… Cada passo acrescenta exatamente 50 amostras, isolando o efeito do batch e permitindo medir o ganho marginal de throughput, a saturação e o limite de memória sem saltos arbitrários.

O batch recomendado é o de maior throughput cuja validação fica a no máximo **0,005** (0,5 ponto percentual) do melhor batch. Início, passo, máximo, tolerância e justificativa ficam gravados no plano auditável.

In [ ]:
RUN_BATCH_SWEEP = True
# Escada uniforme: +50 por tentativa mede o ganho marginal de forma comparável.
BATCH_START = 50
BATCH_STEP = 50
BATCH_MAX = 300       # fecha a escada: 50, 100, 150, ..., 300
BATCH_QUALITY_TOLERANCE = .005  # aceita até 0,5 p.p. abaixo da melhor validação
BATCH_SWEEP_EPOCHS = 10

batch_schedule = list(range(BATCH_START, BATCH_MAX + 1, BATCH_STEP))
assert batch_schedule[-1] == BATCH_MAX, "BATCH_MAX deve fechar a escada."
print(f"Escada de batches: {batch_schedule} | tolerância: {BATCH_QUALITY_TOLERANCE:.1%}")

if RUN_BATCH_SWEEP:
    if 'hpo_table' not in globals():
        hpo_table, _ = automated_hpo(config, trials_per_cnn=8, search_epochs=globals().get('HPO_SEARCH_EPOCHS', 10), backend='thread')
    batch_sweeps = run_batch_sweeps_for_hpo_winners(
        hpo_table,
        batch_start=BATCH_START,
        batch_max=BATCH_MAX,
        step=BATCH_STEP,
        epochs=BATCH_SWEEP_EPOCHS,
        quality_tolerance=BATCH_QUALITY_TOLERANCE,
        backend='thread',
    )
    for cnn, sweep in batch_sweeps.items():
        print(f"{cnn.upper()} | artefatos: {sweep['directory']}")
        figure = plot_batch_sweep(sweep)
        display(figure)
        plt.close(figure)
        print('Batch recomendado:', sweep['winner'])
    show_glossary('batch')
else:
    print('Sweep desativado. Defina RUN_BATCH_SWEEP = True para testar batches de 50 em 50 nos vencedores HPO.')


## 9. Observar e comparar resultados

### 9.1 Registro local e promoção de modelos

Esta célula lê os runs já gravados, gera painéis comparativos e alertas quando houver. Após uma execução, filtre pelo `run_id` retornado para inspecionar um experimento específico.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

runs = filter_runs(discover_runs(), dataset=config.dataset, include_incomplete=True)
summary = runs_summary(runs)
print(summary)

if not runs:
    print('Nenhum run encontrado para os filtros selecionados.')

display(plot_runs_dashboard(runs))


all_runs_frame = load_all_audited_runs()
all_figures = all_run_figures(all_runs_frame)
for plot_key, figure in all_figures.items():
    display(figure)
    show_all_run_note(plot_key)
    plt.close(figure)
all_runs_pdf, _, _ = generate_all_runs_pdf()
print(f"Relatório visual consolidado de todos os runs: {all_runs_pdf}")
plt.close('all')


## 10. Experimentos do TCC

Cada experimento do trabalho é uma função `experimento_NN`, num módulo próprio em
`hsi/experimentos/` (`e06_segmentadores.py`, `e13_pool_quente.py`…). A célula abaixo só
importa o índice — nada roda ao abrir o notebook.

Para executar, use a célula final da seção, ou, sem notebook:

```bash
.venv/bin/python experimentos/executar.py 13 IP
```

Os resultados vão para `experimentos/resultados/` e toda execução longa é retomável:
o CSV é a fonte de verdade, e as células já medidas são puladas.


In [ ]:
# O código desta seção está em:
#     hsi/experimentos/__init__.py
#     hsi/experimentos/e05_cnn_fixa.py
#     hsi/experimentos/e06_segmentadores.py
#     hsi/experimentos/e08_estrategias.py
#     hsi/experimentos/e09_validacao_regiao.py
#     hsi/experimentos/e10_impacto_hpo.py
#     hsi/experimentos/e11_particao.py
#     hsi/experimentos/e12_particao_split.py
#     hsi/experimentos/e13_pool_quente.py
#     hsi/experimentos/infra.py
#     hsi/experimentos/medicoes.py

from hsi.experimentos import EXPERIMENTOS

In [ ]:
# ============================================================================
# EXECUTAR UM EXPERIMENTO
# ============================================================================
# Troque `EXECUTAR` pelo número desejado e rode esta célula. `None` não faz nada,
# que é o estado em que o notebook fica guardado.
#
# Os experimentos 11, 12 e 13 usam o executor de processos. Como as funções agora
# moram no pacote `hsi`, o `spawn` as encontra também a partir do kernel; para
# medir tempo sem o kernel competindo por CPU, prefira o terminal:
#
#     .venv/bin/python experimentos/executar.py 13 IP
#
EXECUTAR = None
CENAS_DO_EXPERIMENTO = ("IP",)

if EXECUTAR is not None:
    EXPERIMENTOS[str(EXECUTAR).zfill(2)](*CENAS_DO_EXPERIMENTO)


## Próximos ajustes

Para reduzir consumo de memória, diminua `batch_size` ou aumente `gradient_accumulation_steps`. Para promover um modelo, use `ModelRegistry(output_dir).promote(run_id)` após conferir as métricas.

Para ampliar a busca de hiperparâmetros, edite `HPO_SPACE` na seção 5.4 — acrescentar um eixo é acrescentar uma chave, e `hpo_search_space` já registra o espaço efetivo de cada CNN. Um hipertunador guiado (TPE/bayesiano) ou um algoritmo evolutivo entrariam substituindo apenas `hpo_candidates`, que é o único ponto que decide *quais* configurações treinar; `automated_hpo` e toda a auditoria seguem iguais.